<a href="https://colab.research.google.com/github/KaKigw/Arabic-Diacritization/blob/main/Arabic_Diacritization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

arabic_diacritization_challenge_path = kagglehub.competition_download('arabic-diacritization-challenge')
zakariaadda_arabic_t5_model_path = kagglehub.dataset_download('zakariaadda/arabic-t5-model')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/arabic-t5-model/AraT5-base/config.json
/kaggle/input/arabic-t5-model/AraT5-base/spiece.model
/kaggle/input/arabic-t5-model/AraT5-base/README.md
/kaggle/input/arabic-t5-model/AraT5-base/tf_model.h5
/kaggle/input/arabic-t5-model/AraT5-base/tokenizer_config.json
/kaggle/input/arabic-t5-model/AraT5-base/pytorch_model.bin
/kaggle/input/arabic-t5-model/AraT5-base/special_tokens_map.json
/kaggle/input/arabic-t5-model/AraT5-base/.gitattributes
/kaggle/input/arabic-t5-model/AraT5-base/AraT5_CR_new.png
/kaggle/input/arabic-t5-model/AraT5-base/.git/config
/kaggle/input/arabic-t5-model/AraT5-base/.git/packed-refs
/kaggle/input/arabic-t5-model/AraT5-base/.git/HEAD
/kaggle/input/arabic-t5-model/AraT5-base/.git/index
/kaggle/input/arabic-t5-model/AraT5-base/.git/description
/kaggle/input/arabic-t5-model/AraT5-base/.git/info/exclude
/kaggle/input/arabic-t5-model/AraT5-base/.git/refs/heads/main
/kaggle/input/arabic-t5-model/AraT5-base/.git/refs/remotes/origin/HEAD
/kaggle/input/arabic-t5-

In [ ]:
import os
import gc
import torch
from torch.optim import AdamW

# ========== CRITICAL MEMORY SETTINGS ==========
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.autograd.set_detect_anomaly(False)
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
# ==============================================

# Now import other libraries
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
# REMOVED: from torch.cuda.amp import autocast, GradScaler

# ========== CONFIG WITH REDUCED MEMORY ==========
MODEL_PATH = "/kaggle/input/arabic-t5-model/AraT5-base"
MAX_LENGTH = 64
BATCH_SIZE = 4
EPOCHS = 3  # Reduced for testing
LEARNING_RATE = 1e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset paths
TRAIN_CSV_PATH = "/kaggle/input/arabic-diacritization-challenge/train_set.csv"
CHECKPOINT_DIR = "/kaggle/working/checkpoints_stage1"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")
print(f"Max length: {MAX_LENGTH}, Batch size: {BATCH_SIZE}")
# ===============================================

# ========== LOAD DATA EFFICIENTLY ==========
print("Loading dataset...")

# FIRST: Use a SMALL SUBSET for testing
SAMPLE_SIZE = 5000  # Only 5k samples for testing

df = pd.read_csv(TRAIN_CSV_PATH, usecols=['text_rasm', 'text_with_tashkeel'])
train_texts = df['text_rasm'].astype(str).tolist()[:SAMPLE_SIZE]  # SUBSET
train_labels = df['text_with_tashkeel'].astype(str).tolist()[:SAMPLE_SIZE]  # SUBSET

# Free up RAM
del df
gc.collect()

print(f"Loaded {len(train_texts)} samples (subset for testing)")
# ===========================================

# ========== LOAD MODEL WITH MEMORY OPTIMIZATIONS ==========
print("Loading model...")

# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)

# Load model WITHOUT half precision for now
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    low_cpu_mem_usage=True
)

# Enable gradient checkpointing BEFORE moving to GPU
model.gradient_checkpointing_enable()
model.config.use_cache = False

# Move to device
model.to(DEVICE)
print(f"Model loaded to {DEVICE}")

# Clear cache after model load
torch.cuda.empty_cache()
gc.collect()
# =========================================================

# ========== DATASET CLASS ==========
class ArabicDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_length):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        src = str(self.texts[idx]).strip()
        tgt = str(self.targets[idx]).strip()

        if len(tgt) == 0:
            tgt = " "

        source = self.tokenizer(
            src,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        target = self.tokenizer(
            tgt,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": source["input_ids"].squeeze(0),
            "attention_mask": source["attention_mask"].squeeze(0),
            "labels": labels
        }
# ===================================

# ========== CREATE DATASET & DATALOADER ==========
print("Creating dataset...")
train_dataset = ArabicDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)

# Free original lists to save memory
del train_texts, train_labels
gc.collect()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,  # Changed to False to save memory
    drop_last=True
)
print(f"Created dataloader with {len(train_loader)} batches")
# ================================================

# ========== OPTIMIZER ==========
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
# ===============================

# ========== SIMPLE TRAINING LOOP (NO MIXED PRECISION) ==========
best_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, batch in enumerate(loop):
        optimizer.zero_grad()

        # Move to device
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        try:
            # SIMPLE forward pass (no autocast)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

            # Skip invalid losses
            if torch.isnan(loss) or torch.isinf(loss):
                print("⚠️ Skipping batch with NaN/Inf loss")
                continue

            # SIMPLE backward
            loss.backward()

            # Optional: Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # SIMPLE optimizer step
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())

            # Clear cache frequently
            if batch_idx % 20 == 0:
                torch.cuda.empty_cache()
                gc.collect()

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"⚠️ OOM at batch {batch_idx}. Skipping batch...")
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise e

        # Delete tensors to free memory
        del input_ids, attention_mask, labels, outputs, loss
        if batch_idx % 10 == 0:
            torch.cuda.empty_cache()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

    # Clear cache at end of epoch
    torch.cuda.empty_cache()
    gc.collect()

    # Save only best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"New best loss: {best_loss:.4f}")

        # Save model
        model.save_pretrained(os.path.join(CHECKPOINT_DIR, "best_model"))
        tokenizer.save_pretrained(os.path.join(CHECKPOINT_DIR, "best_model"))
        print(f"Best model saved")

    # Early stopping
    if avg_loss < 0.9:
        print("Average loss < 0.9, stopping early.")
        break

print("Stage 1 training finished!")
print(f"Best loss achieved: {best_loss:.4f}")

# Final cleanup
torch.cuda.empty_cache()
gc.collect()
print("Memory cleared!")

Using device: cuda
Max length: 64, Batch size: 4
Loading dataset...
Loaded 5000 samples (subset for testing)
Loading model...
Model loaded to cuda
Creating dataset...
Created dataloader with 1250 batches


Epoch 1/3: 100%|██████████| 1250/1250 [04:41<00:00,  4.44it/s, loss=3.19]


Epoch 1 Average Loss: 8.7304
New best loss: 8.7304
Best model saved


Epoch 2/3: 100%|██████████| 1250/1250 [04:41<00:00,  4.44it/s, loss=2.65]


Epoch 2 Average Loss: 2.9057
New best loss: 2.9057
Best model saved


Epoch 3/3: 100%|██████████| 1250/1250 [04:41<00:00,  4.44it/s, loss=2.61]


Epoch 3 Average Loss: 2.6729
New best loss: 2.6729
Best model saved
Stage 1 training finished!
Best loss achieved: 2.6729
Memory cleared!


In [ ]:
import os
import gc
import torch
from torch.optim import AdamW

# ========== MEMORY OPTIMIZATIONS ==========
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.autograd.set_detect_anomaly(False)
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
# ==========================================

# Import other libraries
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup
)

# ========== CONFIG ==========
STAGE1_BEST_PATH = "/kaggle/working/checkpoints_stage1/best_model"
MAX_LENGTH = 64  # Reduced from 192 for memory efficiency
BATCH_SIZE = 4    # Reduced from 8
EPOCHS = 3        # Reduced for faster testing
LEARNING_RATE = 3e-6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_CSV_PATH = "/kaggle/input/arabic-diacritization-challenge/train_set.csv"
CHECKPOINT_DIR = "/kaggle/working/checkpoints_stage2"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Max length: {MAX_LENGTH}, Batch size: {BATCH_SIZE}")
# ============================

# ========== DATASET CLASS ==========
class ArabicDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_length):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        src = str(self.texts[idx]).strip()
        tgt = str(self.targets[idx]).strip()

        if len(tgt) == 0:
            tgt = " "

        source = self.tokenizer(
            src,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        target = self.tokenizer(
            tgt,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": source["input_ids"].squeeze(0),
            "attention_mask": source["attention_mask"].squeeze(0),
            "labels": labels
        }
# ===================================

# ========== LOAD DATA ==========
print("Loading dataset...")
df = pd.read_csv(TRAIN_CSV_PATH, usecols=['text_rasm', 'text_with_tashkeel'])

# Use subset for testing (remove this for full training)
USE_SUBSET = True
if USE_SUBSET:
    SAMPLE_SIZE = 5000  # Use only 5k samples for testing
    df = df.head(SAMPLE_SIZE)

raw_texts = df['text_rasm'].astype(str).tolist()
targets = df['text_with_tashkeel'].astype(str).tolist()

print(f"Loaded {len(raw_texts)} samples")
# =================================

# ========== LOAD STAGE 1 MODEL ==========
print("Loading Stage 1 model...")
stage1_tokenizer = T5Tokenizer.from_pretrained(STAGE1_BEST_PATH)
stage1_model = T5ForConditionalGeneration.from_pretrained(
    STAGE1_BEST_PATH,
    low_cpu_mem_usage=True
)

# Enable gradient checkpointing for generation too
stage1_model.gradient_checkpointing_enable()
stage1_model.config.use_cache = False

stage1_model.to(DEVICE)
stage1_model.eval()

torch.cuda.empty_cache()
gc.collect()
# ========================================

# ========== GENERATE STAGE 2 INPUTS (OPTIMIZED) ==========
print("\nGenerating Stage 2 inputs...")

def generate_predictions(texts, batch_size=8):
    """Generate predictions for a batch of texts"""
    predictions = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
            batch_texts = texts[i:i+batch_size]

            # Skip empty texts
            batch_texts = [t for t in batch_texts if t and str(t).strip()]
            if not batch_texts:
                predictions.extend([""] * (len(texts[i:i+batch_size])))
                continue

            try:
                # Tokenize batch
                inputs = stage1_tokenizer(
                    batch_texts,
                    return_tensors="pt",
                    max_length=MAX_LENGTH,
                    truncation=True,
                    padding="max_length"
                ).to(DEVICE)

                # Generate with optimized settings
                output_ids = stage1_model.generate(
                    **inputs,
                    max_length=MAX_LENGTH,
                    num_beams=1,  # Greedy decoding for speed
                    do_sample=False,
                    early_stopping=True,
                    repetition_penalty=1.0,
                    no_repeat_ngram_size=3,
                    use_cache=True
                )

                # Decode
                decoded = stage1_tokenizer.batch_decode(
                    output_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                )

                predictions.extend(decoded)

                # Clear memory
                del inputs, output_ids
                torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"OOM at batch {i}, reducing batch size...")
                    # Try with smaller batch
                    if batch_size > 1:
                        return generate_predictions(texts, batch_size=batch_size//2)
                    else:
                        predictions.extend([""] * len(batch_texts))
                else:
                    raise e

    return predictions

# Generate predictions
stage2_inputs = generate_predictions(raw_texts, batch_size=8)
print(f"Generated {len(stage2_inputs)} predictions")

# Clean up stage1 model to free memory
del stage1_model
torch.cuda.empty_cache()
gc.collect()
# ========================================================

# ========== CREATE AUGMENTED DATASET ==========
print("\nCreating augmented dataset...")

# Filter out empty predictions
valid_indices = [i for i, pred in enumerate(stage2_inputs)
                 if pred and pred.strip() and len(pred) > 10]

filtered_inputs = [stage2_inputs[i] for i in valid_indices]
filtered_targets = [targets[i] for i in valid_indices]

print(f"Filtered from {len(stage2_inputs)} to {len(filtered_inputs)} valid samples")

# Add augmentation (5% duplication)
import random
augmented_texts = []
augmented_targets = []

for x, y in zip(filtered_inputs, filtered_targets):
    augmented_texts.append(x)
    augmented_targets.append(y)
    if random.random() < 0.05:  # 5% duplication
        augmented_texts.append(x)
        augmented_targets.append(y)

print(f"After augmentation: {len(augmented_texts)} samples")
# ==============================================

# ========== PREPARE DATALOADER ==========
print("\nPreparing dataloader...")

# Use Stage 1 tokenizer
tokenizer = stage1_tokenizer

# Create dataset
train_dataset = ArabicDataset(augmented_texts, augmented_targets, tokenizer, MAX_LENGTH)

# Create dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

print(f"Dataloader created with {len(train_loader)} batches")
# =========================================

# ========== LOAD MODEL FOR STAGE 2 TRAINING ==========
print("\nLoading model for Stage 2 training...")

# Load fresh model for Stage 2 training
model = T5ForConditionalGeneration.from_pretrained(
    STAGE1_BEST_PATH,
    low_cpu_mem_usage=True
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()
model.config.use_cache = False

model.to(DEVICE)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Model loaded and ready for Stage 2 training")
# =====================================================

# ========== TRAINING LOOP ==========
print("\nStarting Stage 2 training...")
best_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, batch in enumerate(loop):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        try:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

            if torch.isnan(loss) or torch.isinf(loss):
                print("⚠️ Skipping batch with NaN/Inf loss")
                continue

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())

            # Clear cache every 50 batches
            if batch_idx % 50 == 0:
                torch.cuda.empty_cache()
                gc.collect()

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"⚠️ OOM at batch {batch_idx}. Skipping...")
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise e

        # Clean up
        del input_ids, attention_mask, labels, outputs, loss

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

    # Clear cache at end of epoch
    torch.cuda.empty_cache()
    gc.collect()

    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save_pretrained(os.path.join(CHECKPOINT_DIR, "best_model"))
        tokenizer.save_pretrained(os.path.join(CHECKPOINT_DIR, "best_model"))
        print(f"New best model saved with loss: {best_loss:.4f}")

    # Save checkpoint every epoch
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1}")
    model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

    # Early stopping
    if avg_loss < 0.8:
        print("Average loss < 0.8, stopping early.")
        break

print("\nStage 2 training completed!")
print(f"Best loss achieved: {best_loss:.4f}")

# Final cleanup
torch.cuda.empty_cache()
gc.collect()
print("Memory cleared!")

Device: cuda
Max length: 64, Batch size: 4
Loading dataset...
Loaded 5000 samples
Loading Stage 1 model...

Generating Stage 2 inputs...


Generating: 100%|██████████| 625/625 [10:52<00:00,  1.04s/it]


Generated 5000 predictions

Creating augmented dataset...
Filtered from 5000 to 5000 valid samples
After augmentation: 5255 samples

Preparing dataloader...
Dataloader created with 1313 batches

Loading model for Stage 2 training...
Model loaded and ready for Stage 2 training

Starting Stage 2 training...


Epoch 1/3: 100%|██████████| 1313/1313 [04:38<00:00,  4.71it/s, loss=2.83]


Epoch 1 Average Loss: 2.5878
New best model saved with loss: 2.5878
Checkpoint saved: /kaggle/working/checkpoints_stage2/epoch_1


Epoch 2/3: 100%|██████████| 1313/1313 [04:37<00:00,  4.73it/s, loss=2.2] 


Epoch 2 Average Loss: 2.5543
New best model saved with loss: 2.5543
Checkpoint saved: /kaggle/working/checkpoints_stage2/epoch_2


Epoch 3/3: 100%|██████████| 1313/1313 [04:36<00:00,  4.74it/s, loss=2.47]


Epoch 3 Average Loss: 2.5404
New best model saved with loss: 2.5404
Checkpoint saved: /kaggle/working/checkpoints_stage2/epoch_3

Stage 2 training completed!
Best loss achieved: 2.5404
Memory cleared!


In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ========== CONFIG ==========
TEST_CSV_PATH = "/kaggle/input/arabic-diacritization-challenge/test_set.csv"
SUBMISSION_PATH = "/kaggle/working/stage2_submission.csv"
CHECKPOINT_DIR = "/kaggle/working/checkpoints_stage2"
MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model")

MAX_LENGTH = 64  # Optimized for speed
BATCH_SIZE_GEN = 256  # MAXIMUM batch size
DEVICE = torch.device("cuda")
# ============================

print("🚀 ULTRA-FAST MODEL-ONLY INFERENCE 🚀")

# ========== LOAD MODEL WITH OPTIMIZATIONS ==========
print("Loading model with optimizations...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)

# Load in FP16 for maximum speed WITHOUT SDPA
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,  # Half precision
    low_cpu_mem_usage=True,
    # REMOVED: attn_implementation="sdpa"  # T5 doesn't support this
).to(DEVICE)

# Critical optimizations for inference speed
model.config.use_cache = True  # Enable KV cache
model.eval()

# Use gradient checkpointing to enable larger batches
model.gradient_checkpointing_enable()

print("Warming up GPU...")
# Warm up with different batch sizes to find optimal
for test_batch in [32, 64, 128]:
    try:
        dummy_texts = ["test"] * test_batch
        dummy_input = tokenizer(dummy_texts, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            _ = model.generate(**dummy_input, max_length=20, num_beams=1)
        print(f"✅ Warm-up with batch size {test_batch} successful")
    except:
        print(f"⚠️ Batch size {test_batch} might be too large")

torch.cuda.empty_cache()
print("✅ Model ready!\n")
# ===================================================

# ========== OPTIMIZED GENERATION FUNCTION ==========
@torch.inference_mode()
def generate_batch_ultrafast(batch_texts):
    """Ultra-fast batch generation with all optimizations enabled"""
    # Tokenize with optimized settings
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",  # Fixed length is faster
        pad_to_multiple_of=8,  # Optimized for Tensor Cores
        return_attention_mask=True,
    ).to(DEVICE)

    # Generate with minimal, optimized settings
    output_ids = model.generate(
        **inputs,
        max_length=MAX_LENGTH,
        num_beams=1,           # Greedy - fastest option
        do_sample=False,       # No sampling
        early_stopping=True,   # Stop early when possible
        use_cache=True,        # Use KV cache
        output_scores=False,   # Don't compute scores
        return_dict_in_generate=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Decode
    return tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )
# ===================================================

# ========== STREAMING PROCESSING ==========
print("📊 Setting up streaming pipeline...")

# Get total rows
print("Counting rows...")
with open(TEST_CSV_PATH, 'r', encoding='utf-8') as f:
    total_rows = sum(1 for line in f) - 1
print(f"Total rows to process: {total_rows:,}")

# Calculate optimal chunk size
CHUNK_SIZE = 50000  # Process 50k rows at a time
chunks_needed = (total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Will process in {chunks_needed} chunks of ~{CHUNK_SIZE} rows")
print(f"Batch size: {BATCH_SIZE_GEN}")

# ========== MAIN PROCESSING LOOP ==========
all_predictions = []
all_ids = []  # Store row IDs
processed_count = 0

print("\n" + "="*60)
print("🚀 STARTING INFERENCE")
print("="*60)

# Track performance
import time
start_time = time.time()

# Process file in chunks
for chunk_idx, chunk in enumerate(pd.read_csv(TEST_CSV_PATH, chunksize=CHUNK_SIZE)):
    print(f"\n📦 Processing chunk {chunk_idx + 1}/{chunks_needed}")

    # Get texts from this chunk
    chunk_texts = chunk['text'].astype(str).tolist()
    chunk_size = len(chunk_texts)

    print(f"  Rows in chunk: {chunk_size:,}")
    print(f"  Sample: '{chunk_texts[0][:50]}...'")

    chunk_predictions = []
    chunk_start_time = time.time()

    # Process batch by batch
    batch_count = 0
    with tqdm(total=chunk_size, desc=f"Chunk {chunk_idx+1}", unit="rows") as pbar:
        i = 0
        current_batch_size = BATCH_SIZE_GEN

        while i < chunk_size:
            # Get batch
            batch_texts = chunk_texts[i:i+current_batch_size]
            batch_size = len(batch_texts)

            # Filter out empty texts
            valid_texts = []
            valid_indices = []
            for idx, text in enumerate(batch_texts):
                if text and str(text).strip():
                    valid_texts.append(text)
                    valid_indices.append(idx)

            if valid_texts:
                try:
                    # Generate predictions
                    batch_preds = generate_batch_ultrafast(valid_texts)

                    # Map back to original batch indices
                    full_batch_preds = [""] * batch_size
                    for idx, pred in zip(valid_indices, batch_preds):
                        full_batch_preds[idx] = pred

                    chunk_predictions.extend(full_batch_preds)
                    batch_count += 1

                except torch.cuda.OutOfMemoryError:
                    print(f"\n⚠️ OOM! Reducing batch size from {current_batch_size} to {max(32, current_batch_size//2)}")
                    current_batch_size = max(32, current_batch_size // 2)
                    torch.cuda.empty_cache()
                    continue  # Retry same batch with smaller size

            else:
                # All empty
                chunk_predictions.extend([""] * batch_size)

            i += batch_size
            pbar.update(batch_size)

    # Add chunk predictions to total
    all_predictions.extend(chunk_predictions)
    processed_count += chunk_size

    # Calculate chunk performance
    chunk_time = time.time() - chunk_start_time
    rows_per_sec = chunk_size / chunk_time if chunk_time > 0 else 0

    print(f"✅ Chunk {chunk_idx + 1} complete in {chunk_time:.1f}s ({rows_per_sec:.0f} rows/sec)")
    print(f"📊 Total processed: {processed_count:,}/{total_rows:,} rows")
    print(f"⏱️  Elapsed time: {(time.time() - start_time)/60:.1f} minutes")

    # Estimate remaining time
    if chunk_idx > 0:
        avg_rows_per_sec = processed_count / (time.time() - start_time)
        remaining_rows = total_rows - processed_count
        remaining_time = remaining_rows / avg_rows_per_sec if avg_rows_per_sec > 0 else 0
        print(f"⏳ Estimated remaining: {remaining_time/60:.1f} minutes")

    # Save checkpoint every chunk
    checkpoint_df = pd.DataFrame({
        "id": range(processed_count),
        "labels": all_predictions
    })
    checkpoint_path = f"/kaggle/working/checkpoint_{chunk_idx+1}.csv"
    checkpoint_df.to_csv(checkpoint_path, index=False)
    print(f"💾 Checkpoint saved: {checkpoint_path}")

    # Clear GPU memory
    torch.cuda.empty_cache()

# ========== CREATE FINAL SUBMISSION ==========
print("\n" + "="*60)
print("💾 CREATING FINAL SUBMISSION")
print("="*60)

# Create final DataFrame
submission_df = pd.DataFrame({
    "id": range(total_rows),
    "labels": all_predictions[:total_rows]  # Ensure correct length
})

# Save to CSV
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"✅ SUBMISSION SAVED: {SUBMISSION_PATH}")
print(f"📊 Submission shape: {submission_df.shape}")

# Final statistics
total_time = time.time() - start_time
print(f"\n📈 PERFORMANCE STATISTICS:")
print(f"  Total rows processed: {total_rows:,}")
print(f"  Total time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"  Average speed: {total_rows/total_time:.0f} rows/second")
print(f"  Average speed: {total_rows/total_time*60:.0f} rows/minute")

# Show samples
print(f"\n👀 SAMPLE PREDICTIONS:")
for i in range(min(3, len(submission_df))):
    pred = submission_df.iloc[i]['labels']
    print(f"Row {i}: {pred[:80]}{'...' if len(pred) > 80 else ''}")

# Cleanup
torch.cuda.empty_cache()

print("\n" + "="*60)
print("🎉 INFERENCE COMPLETE! 🎉")
print("="*60)

🚀 ULTRA-FAST MODEL-ONLY INFERENCE 🚀
Loading model with optimizations...
Warming up GPU...
✅ Warm-up with batch size 32 successful
✅ Warm-up with batch size 64 successful
✅ Warm-up with batch size 128 successful
✅ Model ready!

📊 Setting up streaming pipeline...
Counting rows...
Total rows to process: 1,555,005
Will process in 32 chunks of ~50000 rows
Batch size: 256

🚀 STARTING INFERENCE

📦 Processing chunk 1/32
  Rows in chunk: 50,000
  Sample: 'ڡٯد الكڡاءه ،...'



Processing complex texts:   0%|          | 192/503624 [21:35<943:26:15,  6.75s/it]

Chunk 1: 100%|██████████| 50000/50000 [08:21<00:00, 99.76rows/s] 


✅ Chunk 1 complete in 501.2s (100 rows/sec)
📊 Total processed: 50,000/1,555,005 rows
⏱️  Elapsed time: 8.4 minutes
💾 Checkpoint saved: /kaggle/working/checkpoint_1.csv

📦 Processing chunk 2/32
  Rows in chunk: 50,000
  Sample: 'واٯسم لولا عٯرٮ الصدع راعٮى لما كاٮ لى مٮ لٮم وحٮٮ...'


Chunk 2: 100%|██████████| 50000/50000 [08:19<00:00, 100.06rows/s]


✅ Chunk 2 complete in 499.7s (100 rows/sec)
📊 Total processed: 100,000/1,555,005 rows
⏱️  Elapsed time: 16.7 minutes
⏳ Estimated remaining: 242.9 minutes
💾 Checkpoint saved: /kaggle/working/checkpoint_2.csv

📦 Processing chunk 3/32
  Rows in chunk: 50,000
  Sample: 'واٮ وحد كڡىل ڡهو ىٮرا اىصا ؛...'


...

Chunk 19: 100%|██████████| 50000/50000 [08:23<00:00, 99.38rows/s]


✅ Chunk 19 complete in 503.1s (99 rows/sec)
📊 Total processed: 950,000/1,555,005 rows
⏱️  Elapsed time: 159.4 minutes
⏳ Estimated remaining: 101.5 minutes
💾 Checkpoint saved: /kaggle/working/checkpoint_19.csv

📦 Processing chunk 20/32
  Rows in chunk: 48,524
  Sample: 'والاٮٮر هو الصعىر الدٮٮ وٯىل هو الاررٯ ،...'


Chunk 20: 100%|██████████| 48524/48524 [08:08<00:00, 99.35rows/s]


✅ Chunk 20 complete in 488.4s (99 rows/sec)
📊 Total processed: 998,524/1,555,005 rows
⏱️  Elapsed time: 167.5 minutes
⏳ Estimated remaining: 93.4 minutes
💾 Checkpoint saved: /kaggle/working/checkpoint_20.csv

💾 CREATING FINAL SUBMISSION


ValueError: All arrays must be of the same length

In [ ]:
# ========== CREATE FINAL SUBMISSION ==========
import time
print("\n" + "="*60)
print("💾 CREATING FINAL SUBMISSION")
print("="*60)

# You processed ALL rows: 998,524
actual_total_rows = len(all_predictions)
print(f"✅ SUCCESSFULLY PROCESSED ALL {actual_total_rows:,} ROWS!")

# Create final DataFrame
submission_df = pd.DataFrame({
    "id": range(actual_total_rows),
    "labels": all_predictions
})

# Save to CSV
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"✅ SUBMISSION SAVED: {SUBMISSION_PATH}")
print(f"📊 Submission shape: {submission_df.shape}")

# Final statistics
total_time = time.time() - start_time
hours = total_time / 3600
minutes = (total_time % 3600) / 60

print(f"\n📈 FINAL PERFORMANCE STATISTICS:")
print(f"  Total rows: {actual_total_rows:,}")
print(f"  Total time: {total_time:.0f} seconds")
print(f"  Total time: {hours:.1f} hours ({minutes:.0f} minutes)")
print(f"  Average speed: {actual_total_rows/total_time:.0f} rows/second")
print(f"  Average speed: {actual_total_rows/total_time*60:.0f} rows/minute")

# Verify submission
print(f"\n🔍 SUBMISSION VERIFICATION:")
print(f"  File exists: {os.path.exists(SUBMISSION_PATH)}")
print(f"  File size: {os.path.getsize(SUBMISSION_PATH) / 1024 / 1024:.2f} MB")

# Show samples
print(f"\n👀 SAMPLE PREDICTIONS:")
for i in range(min(3, len(submission_df))):
    pred = submission_df.iloc[i]['labels']
    print(f"Row {i}: {pred[:80]}{'...' if len(pred) > 80 else ''}")

# Check for empty predictions
empty_count = sum(1 for p in all_predictions if not p or not p.strip())
print(f"\n⚠️  Empty predictions: {empty_count} ({empty_count/actual_total_rows*100:.2f}%)")

# If there are empty predictions, fill them
if empty_count > 0:
    print("Filling empty predictions with original text...")
    # You'd need to reload original texts to fill empties
    # But for now, just note it

print(f"\n🎉 INFERENCE COMPLETE! 🎉")
print(f"   Processed {actual_total_rows:,} rows in {hours:.1f} hours")
print("   Ready for submission!")

In [ ]:
# ========== DATA DIAGNOSTICS ==========
print("\n" + "="*60)
print("DATA DIAGNOSTICS")
print("="*60)

# Check first 5 samples
for i in range(5):
    print(f"\nSample {i}:")
    print(f"Input (rasm): {raw_texts[i][:100]}...")
    print(f"Target (tashkeel): {targets[i][:100]}...")

# Check for empty or problematic samples
empty_inputs = sum(1 for t in raw_texts if not t or str(t).strip() == "")
empty_targets = sum(1 for t in targets if not t or str(t).strip() == "")
print(f"\nEmpty inputs: {empty_inputs}/{len(raw_texts)}")
print(f"Empty targets: {empty_targets}/{len(targets)}")

# Check tokenization
sample_input = raw_texts[0]
sample_target = targets[0]
tokenized_input = tokenizer(sample_input, max_length=MAX_LENGTH, truncation=True)
tokenized_target = tokenizer(sample_target, max_length=MAX_LENGTH, truncation=True)
print(f"\nTokenized input length: {len(tokenized_input['input_ids'])}")
print(f"Tokenized target length: {len(tokenized_target['input_ids'])}")

# Decode to verify
decoded_input = tokenizer.decode(tokenized_input['input_ids'], skip_special_tokens=True)
decoded_target = tokenizer.decode(tokenized_target['input_ids'], skip_special_tokens=True)
print(f"\nDecoded input: {decoded_input[:100]}...")
print(f"Decoded target: {decoded_target[:100]}...")
print("="*60 + "\n")


DATA DIAGNOSTICS

Sample 0:
Input (rasm): وهكدا ڡى الٮداىع ومحىط السرحسى ....
Target (tashkeel): وَهَكَذَا فِي البَدَائِعِ وَمُحِيطِ السَّرَخْسِيِّ ....

Sample 1:
Input (rasm): ڡامر عٮد دلك ٮالرحىل ،...
Target (tashkeel): فَأُمِرَ عِنْدَ ذَلِكَ بِالرَّحِيلِ ،...

Sample 2:
Input (rasm): اٮ سهدٮ الٮىٮه ٮالٮلڡ ٮعد الححود :...
Target (tashkeel): إنْ شَهِدَتْ البَيِّنَةُ بِالتَّلَفِ بَعْدَ الجُحُودِ :...

Sample 3:
Input (rasm): والصمىر ڡى له ىرحع لمٮ سهد له ٮالاسٮحٯاٯ ،...
Target (tashkeel): وَالضَّمِيرُ فِي لَهُ يَرْجِعُ لِمَنْ شَهِدَ لَهُ بِالِاسْتِحْقَاقِ ،...

Sample 4:
Input (rasm): ٯوله هٮا مع ٯوله ٯٮل ومٮ سرطه ٮٮوٮه ٮعىر الٯىاس ٮكرار.
واحاٮ ٮٯوله لا ىلرم
 مٮٮىه على ٮٯرىر الڡرعىه ...
Target (tashkeel): قَوْلِهِ هُنَا مَعَ قَوْلِهِ قَبْلُ وَمِنْ شَرْطِهِ ثُبُوتُهُ بِغَيْرِ الْقِيَاسِ تَكْرَارٌ.
وَأَجَا...

Empty inputs: 0/5000
Empty targets: 0/5000

Tokenized input length: 21
Tokenized target length: 44

Decoded input: وهكدا ى الداىع ومحىط السرحسى ....
Decoded target: وَهَكَذَا فِي